In [1]:

import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split

import warnings
warnings.filterwarnings('ignore')

print('All imports successful.')

All imports successful.


In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f"Training samples: {len(train)}")
print(f"Test samples:     {len(test)}")
print(f"\nLabel distribution:")
print(train['LABEL'].value_counts().sort_index())
print(f"\nSample training row:")
print(train.iloc[1][['LABEL', 'TEXT']].to_string())

Training samples: 70317
Test samples:     17580

Label distribution:
0    32289
1    19139
2    18889
Name: LABEL, dtype: int64

Sample training row:
LABEL                                                    1
TEXT     Just getting released from a six month drug re...


In [4]:
# Remove HTML tags (common in movie reviews e.g. <br /> tags)
def clean_text(text):
    text = str(text)
    text = re.sub(r'<br\s*/?>', ' ', text)  # replace <br> with space
    text = re.sub(r'<[^>]+>', ' ', text)    # remove any other HTML tags
    text = text.lower()                      # lowercase
    text = re.sub(r'\s+', ' ', text).strip() # normalize whitespace
    return text

train['clean_text'] = train['TEXT'].apply(clean_text)
test['clean_text'] = test['TEXT'].apply(clean_text)

# Verify
print("Before:", train.iloc[1]['TEXT'][:80])
print("After: ", train.iloc[1]['clean_text'][:80])

Before: Just getting released from a six month drug rehabilitation program and having se
After:  just getting released from a six month drug rehabilitation program and having se


In [6]:
# Cell 4: TF-IDF Vectorization (Unit 4 - text representations)
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=100000,
    sublinear_tf=True,
    min_df=2
)

X_text = train['clean_text'].fillna('')
X_tr, X_dev, y_tr, y_dev = train_test_split(
    X_text, train['LABEL'], test_size=0.2, random_state=42
)

X_tr_tfidf   = tfidf.fit_transform(X_tr)
X_dev_tfidf  = tfidf.transform(X_dev)
X_test_tfidf = tfidf.transform(test['clean_text'].fillna(''))

print(f"Feature matrix shape: {X_tr_tfidf.shape}")

Feature matrix shape: (56253, 100000)


In [7]:
# Cell 5: Train Logistic Regression (Unit 3)
lr = LogisticRegression(
    C=5.0,
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
    multi_class='multinomial'
)

lr.fit(X_tr_tfidf, y_tr)
y_pred = lr.predict(X_dev_tfidf)

print(f"Dev set accuracy: {accuracy_score(y_dev, y_pred):.4f}")
print()
print(classification_report(y_dev, y_pred,
      target_names=['not a review', 'positive review', 'negative review']))

Dev set accuracy: 0.9339

                 precision    recall  f1-score   support

   not a review       0.97      0.98      0.97      6402
positive review       0.90      0.89      0.90      3850
negative review       0.91      0.90      0.91      3812

       accuracy                           0.93     14064
      macro avg       0.93      0.92      0.92     14064
   weighted avg       0.93      0.93      0.93     14064



In [8]:
# Cell 6: Error analysis (adapted from LING 539 Unit 3 error analysis notebook)
joined = pd.DataFrame({
    'Text': X_dev.values,
    'Label': y_dev.values,
    'Prediction': y_pred
})

misclassified = joined[joined['Label'] != joined['Prediction']]
print(f"Total misclassified: {len(misclassified)} out of {len(y_dev)}")
print()

# Look at a sample of errors
CLASS2NAME = {0: 'not a review', 1: 'positive review', 2: 'negative review'}

def inspect_error(row):
    print(f"True: {CLASS2NAME[row['Label']]} | Predicted: {CLASS2NAME[row['Prediction']]}")
    print(f"Text: {str(row['Text'])[:200]}")
    print()

print("=== Sample misclassified examples ===")
misclassified.sample(n=5, random_state=42).apply(inspect_error, axis=1)

Total misclassified: 930 out of 14064

=== Sample misclassified examples ===
True: positive review | Predicted: negative review
Text: it seems that people are split on these movie, and naturally they all think that the people on the other side are complete morons with brains unfit (or perfectly fit?) for human bodies. this can proba

True: positive review | Predicted: not a review
Text: this is one of my all-time favorites. early christian bale-- great actor!

True: positive review | Predicted: negative review
Text: okay, so ghoulies 4 is kind of bad. and it doesn't really even have the ghoulies in it. and the acting is bad. the storyline is stupid. but i forget to mention how funny this film is. it is so campy, 

True: positive review | Predicted: not a review
Text: they have put ships in the sky to kill people. america is going the back against them.

True: negative review | Predicted: positive review
Text: this show is awesome and we have been enjoying it thoroughly. set in alaska, 

11359    None
9933     None
423      None
5749     None
3054     None
dtype: object

In [9]:
# Cell 7: Generate submission file
# Retrain on full training data before predicting
tfidf_full = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=100000,
    sublinear_tf=True,
    min_df=2
)
X_full = tfidf_full.fit_transform(train['clean_text'].fillna(''))
X_test_full = tfidf_full.transform(test['clean_text'].fillna(''))

lr_full = LogisticRegression(
    C=5.0, max_iter=1000,
    class_weight='balanced',
    solver='lbfgs', multi_class='multinomial'
)
lr_full.fit(X_full, train['LABEL'])

predictions = lr_full.predict(X_test_full)

submission = pd.DataFrame({
    'ID': test['ID'],
    'LABEL': predictions
})
submission.to_csv('submission.csv', index=False)
print(f"Submission saved! Shape: {submission.shape}")
print(submission['LABEL'].value_counts().sort_index())

Submission saved! Shape: (17580, 2)
0    7996
1    4923
2    4661
Name: LABEL, dtype: int64


In [10]:
# Quick check
print(submission.head(10))
print(f"\nExpected shape: (17580, 2)")
print(f"Actual shape:   {submission.shape}")
print(f"\nAny nulls? {submission.isnull().any().any()}")

                     ID  LABEL
0   1087873697464833975      1
1   5853461517618045821      1
2   1225516091890726770      2
3  11795874926011119457      0
4  15956464546963841646      0
5  14751864642209654760      0
6   3005217029968835639      0
7  11376939852998541397      2
8   6837960727095302371      0
9   9201125420407700948      2

Expected shape: (17580, 2)
Actual shape:   (17580, 2)

Any nulls? False
